#### 7. Compare all four ingestion patterns covered (batch CTAS, COPY INTO, Autoloader, Lakeflow Declarative Pipelines) on cost, latency, and operational complexity, and recommend which one Cyntexa should use for a file source that arrives unpredictably throughout the day.

## 1. Batch CTAS (CREATE TABLE AS SELECT)

**Cost:**
- Lowest cost for simple, infrequent loads
- No streaming infrastructure overhead
- May reprocess data unnecessarily if not managed carefully

**Latency:** 
- High latency - manual or scheduled batch execution
- No real-time or near-real-time processing
- Entire dataset typically scanned each run

**Operational Complexity:** 
- Simplest to implement (single SQL statement)
- Manual orchestration required
- No built-in incremental logic or file tracking
- Difficult to handle schema evolution

**Best for:** One-time data loads, small datasets, ad-hoc transformations

---

## 2. COPY INTO

**Cost:** 
- Efficient incremental loading with automatic file tracking
- Only processes new files
- Lower compute cost than full table scans

**Latency:**
- Moderate latency - typically scheduled (minutes to hours)
- Processes files in batches
- Not streaming, but more frequent than CTAS

**Operational Complexity:** 
- Simple SQL syntax with built-in idempotency
- Automatic file tracking (no duplicates)
- Requires scheduling/orchestration for automation
- Limited schema evolution support (strict schema enforcement)
- Manual error handling needed

**Best for:** Scheduled, periodic ingestion of files with stable schemas

---

## 3. Auto Loader (cloudFiles)

**Cost:** 
- Very efficient incremental processing
- File notification mode: minimal overhead for large-scale ingestion
- Directory listing mode: some overhead for small file counts
- Automatic checkpointing prevents reprocessing

**Latency:**
- Near real-time streaming ingestion
- Processes files as they arrive (seconds to minutes)
- Continuous processing with Structured Streaming

**Operational Complexity:**
- Moderate complexity - requires Structured Streaming knowledge
- Automatic schema inference and evolution
- Built-in error handling and rescue data columns
- Checkpoint management handled automatically
- Requires streaming infrastructure (continuous or triggered)

**Best for:** Streaming/near-real-time ingestion, unpredictable file arrivals, schema evolution

---

## 4. Lakeflow Declarative Pipelines (Delta Live Tables)

**Cost:** 
- Higher infrastructure cost (dedicated pipeline compute)
- Enhanced autoscaling and optimization
- Built-in quality monitoring and lineage tracking
- Cost justified for complex pipelines with multiple stages

**Latency:** 
- Near real-time with streaming mode
- Can use Auto Loader underneath
- Automatic dependency management and orchestration

**Operational Complexity:** 
- Declarative syntax simplifies pipeline logic
- Built-in data quality expectations
- Automatic orchestration, retries, and monitoring
- Unified framework for ingestion + transformation
- Steeper learning curve initially
- Less flexibility for custom logic

**Best for:** Production ETL/ELT pipelines with quality requirements, complex multi-stage transformations

---

## Recommendation for Cyntexa

### **Use Auto Loader** for files arriving unpredictably throughout the day

**Rationale:**

1. **Optimal for unpredictable arrivals:** Auto Loader is specifically designed for continuous, event-driven file ingestion. It processes files as they arrive without waiting for scheduled intervals.

2. **Cost-effective:** Only processes new files incrementally with automatic checkpoint management. File notification mode scales efficiently even with high file volumes.

3. **Low latency:** Provides near real-time ingestion (typically within seconds to minutes of file arrival), ensuring data is quickly available for downstream consumption.

4. **Moderate complexity:** While requiring Structured Streaming knowledge, Auto Loader's automatic schema inference, evolution, and error handling significantly reduce operational burden compared to building custom streaming solutions.

5. **Production-ready features:**
   - Automatic exactly-once processing guarantees
   - Schema evolution support (gracefully handles new columns)
   - Rescue data column for malformed records
   - Built-in retry logic and fault tolerance


#### 8. Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run.

In [0]:
%sql
select * from cyntexa_dev.silver.sales_clean

In [0]:
%sql
DESCRIBE HISTORY cyntexa_dev.silver.sales_clean;

-- Time travel to inspect the state before corruption
SELECT COUNT(*), MAX(ingestion_time), MIN(ingestion_time)
FROM cyntexa_dev.silver.sales_clean VERSION AS OF 2;

-- Or use timestamp-based time travel:
SELECT COUNT(*), MAX(ingestion_time), MIN(ingestion_time)
FROM cyntexa_dev.silver.sales_clean TIMESTAMP AS OF '2026-08-26T07:16:07.000+00:00';

-- Verify data quality at the suspected good version
SELECT *
FROM cyntexa_dev.silver.sales_clean VERSION AS OF 2
LIMIT 100;


RESTORE TABLE cyntexa_dev.silver.sales_clean TO VERSION AS OF 2;
-- Or restore by timestamp:
-- RESTORE TABLE cyntexa_dev.silver.sales_clean TO TIMESTAMP AS OF '2026-08-26T07:16:07.000+00:00';

-- Verify the restore succeeded
DESCRIBE HISTORY cyntexa_dev.silver.sales_clean;
-- You should see a new RESTORE operation at the top

-- Validate data quality post-restore
SELECT COUNT(*), MAX(ingestion_time) FROM cyntexa_dev.silver.sales_clean;

-- Check what downstream tables/views depend on silver_sales
SELECT *
FROM system.access.table_lineage
WHERE source_table_full_name = 'cyntexa_dev.silver.sales_clean'
  AND event_time > '2026-08-26T07:16:07.000+00:00';

-- Verify gold layer tables that may need refresh
SELECT COUNT(*) FROM cyntexa_dev.gold.total_revenue_by_products;

-- Identify the bad file from the history
DESCRIBE history cyntexa_dev.silver.sales_clean;


#### 9. (Data Analyst) Using DESCRIBE HISTORY, produce a 'data freshness' report showing how frequently a given table is actually updated, to validate an SLA claim made to a business stakeholder.

In [0]:
%sql
WITH table_history AS (
  SELECT 
    version,
    timestamp,
    operation,
    operationParameters,
    -- Calculate time between updates
    LAG(timestamp) OVER (ORDER BY version DESC) AS previous_update,
    TIMESTAMPDIFF(MINUTE, timestamp, LAG(timestamp) OVER (ORDER BY version DESC)) AS minutes_since_last_update
  FROM (
    DESCRIBE HISTORY cyntexa_dev.silver.sales_clean
  )
  WHERE operation IN ('WRITE', 'MERGE', 'UPDATE', 'DELETE', 'INSERT', 'COPY', 'RESTORE')
),
freshness_stats AS (
  SELECT
    COUNT(*) AS total_updates,
    MIN(timestamp) AS first_update,
    MAX(timestamp) AS last_update,
    TIMESTAMPDIFF(DAY, MIN(timestamp), MAX(timestamp)) AS days_of_history,
    AVG(minutes_since_last_update) AS avg_minutes_between_updates,
    PERCENTILE(minutes_since_last_update, 0.5) AS median_minutes_between_updates,
    MIN(minutes_since_last_update) AS min_minutes_between_updates,
    MAX(minutes_since_last_update) AS max_minutes_between_updates,
    STDDEV(minutes_since_last_update) AS stddev_minutes_between_updates
  FROM table_history
  WHERE minutes_since_last_update IS NOT NULL
)
SELECT
  total_updates,
  first_update,
  last_update,
  days_of_history,
  ROUND(avg_minutes_between_updates / 60, 2) AS avg_hours_between_updates,
  ROUND(median_minutes_between_updates / 60, 2) AS median_hours_between_updates,
  ROUND(min_minutes_between_updates / 60, 2) AS min_hours_between_updates,
  ROUND(max_minutes_between_updates / 60, 2) AS max_hours_between_updates,
  ROUND(stddev_minutes_between_updates / 60, 2) AS stddev_hours_between_updates,
  ROUND(total_updates / NULLIF(days_of_history, 0), 2) AS updates_per_day,
  -- SLA validation: adjust threshold as needed (e.g., 24 hours)
  CASE 
    WHEN avg_minutes_between_updates / 60 <= 24 THEN 'MEETING SLA (≤24h avg)'
    ELSE 'FAILING SLA (>24h avg)'
  END AS sla_status_24h
FROM freshness_stats;




In [0]:
%sql
-- Detailed update timeline for the last 30 days
SELECT
  version,
  timestamp,
  operation,
  TIMESTAMPDIFF(HOUR, LAG(timestamp) OVER (ORDER BY version), timestamp) AS hours_since_last_update
FROM (
  DESCRIBE HISTORY cyntexa_dev.silver.sales_clean
)
WHERE operation IN ('WRITE', 'MERGE', 'UPDATE', 'DELETE', 'INSERT', 'COPY', 'RESTORE', 'CREATE')
  AND timestamp >= CURRENT_TIMESTAMP() - INTERVAL 30 DAYS
ORDER BY version DESC;